<a href="https://colab.research.google.com/github/Teomorales20/SenalesSistemas/blob/master/Simulaciones_de_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp

# 1) Variables simb licas

s, t, L, C, R = sp.symbols('s t L C R',
positive=True, real=True)
Vi = sp.symbols('Vi', real=True, positive=True)
# D(s) fijo en 0.5
D = sp.Rational(1, 2)

In [ ]:
# 2) Funcion de transferencia simbolica
den = L*C*s**2 + (L/R)*s + 1
Hs = Vi*(1 - D)/den # Vo(s) con D=0.5
Hs

In [ ]:
# 3) Respuesta al impulso (inversa de Laplace de H(s))
h_t = sp.inverse_laplace_transform(Hs, s, t)
print("\nRespuesta al impulso h(t):")
print(h_t)

In [ ]:
# 4) Respuesta al escalon (entrada 1/s)
step_s = Hs / s
step_t = sp.inverse_laplace_transform (step_s, s, t)
print("\nRespuesta al escal n y_step(t):")
print(step_t)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# Par metros del circuito
L = 697e-6 # H
C = 22.3e-6 # F
R = 7.0 # ohm
Vi = 12.0 # V
D = 0.5

# Coeficientes del denominador
a2 = L * C
a1 = L / R
a0 = 1.0

# Ganancia efectiva (Vi * (1 - D))
num = [Vi * (1 - D)] # Numerador
den = [a2, a1, a0] # Denominador

system = signal.TransferFunction(num, den)

# Diagrama de Bode
w, mag, phase = signal.bode(system)
plt.figure(figsize=(8,6))
plt.subplot(2,1,1)
plt.semilogx(w, mag)
plt.title('Diagrama de Bode')
plt.ylabel('Magnitud (dB)')
plt.grid(True, which='both', ls=':')

plt.subplot(2,1,2)
plt.semilogx(w, phase)
plt.ylabel('Fase ( )')
plt.xlabel('Frecuencia (rad/s')
plt.grid(True, which='both', ls=':')
plt.tight_layout()
plt.show()

# Respuesta al impulso
t_imp, y_imp = signal.impulse(system)
plt.figure()
plt.plot(t_imp, y_imp)
plt.title('Respuesta al Impulso (D=0.5)')
plt.xlabel('Tiempo (s)')
plt.ylabel('Voltaje de salida (V)')
plt.grid(True)
plt.show()

# Respuesta al escalon
t_step, y_step = signal.step(system)
plt.figure()
plt.plot(t_step, y_step)
plt.title('Respuesta al Escal n (D=0.5)')
plt.xlabel('Tiempo (s)')
plt.ylabel('Voltaje de salida (V)')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parámetros del sistema ---
Vi = 12.0          # Tensión de entrada [V]
L = 697e-6         # Inductancia [H]
C = 22.3e-6        # Capacitancia [F]
R = 7.0            # Resistencia de carga [Ohm]
d_steady = 0.5     # Punto de operación (ciclo de trabajo)

# --- Cálculo de parámetros dinámicos ---
wn = np.sqrt(1 / (L * C))               # Frecuencia natural [rad/s]
zeta = (1 / (2 * R)) * np.sqrt(L / C)   # Factor de amortiguamiento
wd = wn * np.sqrt(1 - zeta**2)          # Frecuencia amortiguada
phi = np.arccos(zeta)                   # Desfase para la ecuación

# --- Tiempo de simulación ---
t = np.linspace(0, 0.003, 1000) # De 0 a 3ms con 1000 puntos

# --- Ecuación analítica de la respuesta al escalón ---
# y(t) = Vi * d * [1 - (e^(-zeta*wn*t) / sqrt(1-zeta^2)) * sin(wd*t + phi)]
v_final = Vi * d_steady
envelope = np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2)
vo = v_final * (1 - envelope * np.sin(wd * t + phi))

# --- Creación de la gráfica ---
plt.figure(figsize=(10, 6))
plt.plot(t * 1000, vo, label='Respuesta $v_o(t)$', color='blue', linewidth=2)
plt.axhline(y=v_final, color='red', linestyle='--', label=f'Valor Final ({v_final}V)')

# Detalles estéticos
plt.title('Respuesta Temporal del Convertidor Buck (Lazo Abierto)', fontsize=14)
plt.xlabel('Tiempo [ms]', fontsize=12)
plt.ylabel('Tensión de Salida [V]', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.ylim(0, 9) # Para ver bien el sobreimpulso

# Mostrar métricas en la gráfica
mp = (np.max(vo) - v_final) / v_final * 100
plt.annotate(f'Sobreimpulso: {mp:.1f}%', xy=(0.4, 7.5), xytext=(0.8, 8.2),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1))

plt.tight_layout()
plt.show()